In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.


ETAPA 0 — Conversão Excel para CSV

In [2]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_origem = "Relatório de Fechamento.xlsx"

df = pd.read_excel(arquivo_origem)

ETAPA 1 — Filtro Fluxo de Caixa

In [3]:
df.columns = df.columns.str.strip()

df = df[df["Objeto"].str.contains("fluxo de caixa", case=False, na=False)]

df.to_excel("relatorio.xlsx", index=False)

print(f"Total de linhas após filtro: {len(df)}")

Total de linhas após filtro: 32943


ETAPA 2 — Tratamento (AY / AZ)

In [4]:
df = pd.read_excel("relatorio.xlsx")

total_inicial = len(df)

col_AY = "Valor Pedido Objeto Corrigido"
col_AZ = "Valor Pedido"

mask_remove = (
    ((df[col_AY].round(2) == 0.01) & (df[col_AZ].round(2) == 0.01)) |
    ((df[col_AY].round(2) == 0.22) & (df[col_AZ].round(2) == 0.22))
)

mask_alerta = (
    (df[col_AY].round(2).isin([0.01, 0.22])) ^
    (df[col_AZ].round(2).isin([0.01, 0.22]))
)

alertas = df[mask_alerta].copy()
alertas.to_excel("alertas_valor.xlsx", index=False) #avisa se um for significativo e outro não

df = df[~(mask_remove | mask_alerta)]

total_final = len(df)

df.to_excel("relatorio.xlsx", index=False)

print("ETAPA 3 concluída ✔")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas enviadas para alertas: {len(alertas)}")
print(f"Total final após tratamento: {total_final}")
print(f"Total removido da base principal: {total_inicial - total_final}")

ETAPA 3 concluída ✔
Total inicial de linhas: 32943
Linhas enviadas para alertas: 1977
Total final após tratamento: 30948
Total removido da base principal: 1995


ETAPA 4 - Excluir por Natureza

In [5]:
import pandas as pd

# Ler relatório original
df = pd.read_excel("relatorio.xlsx")

# =========================
# CONDIÇÃO DE EXCLUSÃO
# =========================

cond_excluir = (
    (
        df["Natureza"].isin([
            "Administrativo - Gerencial",
            "Administrativo - Protestos",
            "Criminal"
        ])
        |
        (df["Natureza Financeira"].astype(str).str.upper() == "NEUTRO")
        |
        (df["Tipo de Ação"].astype(str).str.upper() == "NOTIFICAÇÃO EXTRAJUDICIAL")
        |
        (df["Assunto Principal"].astype(str).str.upper() == "RESCISÃO CONTRATUAL - LOCAÇÃO")
    )
    &
    (df["Valor Pedido Objeto Corrigido"] < 1)
)

# =========================
# APLICAR FILTRO
# =========================

relatorio_limpo = df[~cond_excluir]

# =========================
# SALVAR NOVO RELATÓRIO
# =========================

relatorio_limpo.to_excel("RELATORIO_FILTRADO.xlsx", index=False)

print("Relatório filtrado criado com sucesso.")
print("Registros originais:", len(df))
print("Registros após filtro:", len(relatorio_limpo))
print("Registros removidos:", len(df) - len(relatorio_limpo))

Relatório filtrado criado com sucesso.
Registros originais: 30948
Registros após filtro: 29452
Registros removidos: 1496


ETAPA 4 - Obter o Entradas

In [6]:
import pandas as pd

# Datas do filtro
data_inicio = "2026-01-26"
data_fim = "2026-02-25"

# Ler planilha original
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

# Converter coluna de data
df["Data de cadastro"] = pd.to_datetime(df["Data de cadastro"], errors="coerce")


# =========================
# REMOVER DUPLICADAS
# =========================

df_sem_dup = df.drop_duplicates(subset="Pasta")

# =========================
# FILTRAR ENTRE DATAS
# =========================

filtro = (
    (df_sem_dup["Data de cadastro"] >= data_inicio) &
    (df_sem_dup["Data de cadastro"] <= data_fim)
)

entradas = df_sem_dup[filtro]

# =========================
# SALVAR PLANILHA
# =========================

entradas.to_excel("ENTRADAS.xlsx", index=False)

print("Planilha 'ENTRADAS.xlsx' criada com sucesso.")
print("Total de registros:", len(entradas))

Planilha 'ENTRADAS.xlsx' criada com sucesso.
Total de registros: 18


ETAPA 5 - Entradas Pos BP

In [7]:
import pandas as pd

# Data do filtro
data_bp = "2025-08-25"

# Ler planilha original
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

# Converter coluna de data
df["Data de cadastro"] = pd.to_datetime(df["Data de cadastro"], errors="coerce")


# =========================
# REMOVER DUPLICADAS
# =========================

df_sem_dup = (
    df.sort_values("Data de cadastro", ascending=False)
      .drop_duplicates(subset="Pasta")
)
# =========================
# FILTRAR APÓS BP
# =========================

filtro = df_sem_dup["Data de cadastro"] >= data_bp

pos_bp = df_sem_dup[filtro]

# =========================
# SALVAR PLANILHA
# =========================

pos_bp.to_excel("POS_BP.xlsx", index=False)

print("Planilha 'POS_BP.xlsx' criada com sucesso.")
print("Total de registros:", len(pos_bp))

Planilha 'POS_BP.xlsx' criada com sucesso.
Total de registros: 251


ETAPA 5 — Corte de Data

In [8]:
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

total_inicial = len(df)

col_data = "Data de cadastro"

def conv_data(col):

    # tenta converter datas normais
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    # tenta converter datas Excel
    col_numeric = pd.to_numeric(col, errors="coerce")

    mask_excel = d.isna() & col_numeric.notna()

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    return d

df[col_data] = conv_data(df[col_data])

# ============================================
# >>> ALTERAR AQUI <<<
data_corte_manual = "26/02/2026"
# ============================================

data_corte = pd.to_datetime(data_corte_manual, dayfirst=True)

mask = df[col_data] >= data_corte

linhas_removidas = mask.sum()

df = df[~mask].copy()

total_final = len(df)

df.to_excel("RELATORIO_FILTRADO.xlsx", index=False)

print("ETAPA 5 concluída ✔")
print(f"Data de corte aplicada: {data_corte.date()}")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas removidas (>= data de corte): {linhas_removidas}")
print(f"Total final do documento: {total_final}")

ETAPA 5 concluída ✔
Data de corte aplicada: 2026-02-26
Total inicial de linhas: 29452
Linhas removidas (>= data de corte): 0
Total final do documento: 29452


ETAPA 6 — Separar por Status

In [9]:
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

total_inicial = len(df)

col_status = "Status"

# normalizar texto
df[col_status] = df[col_status].astype(str).str.strip().str.upper()

df_ativos = df[df[col_status] == "ATIVOS"]
df_baixa = df[df[col_status] == "BAIXA PROVISÓRIA"]
df_encerrados = df[df[col_status] == "ENCERRADOS"]

df_outros = df[~df[col_status].isin([
    "ATIVOS",
    "BAIXA PROVISÓRIA",
    "ENCERRADOS"
])]

df_ativos.to_excel("ATIVOS.xlsx", index=False)
df_baixa.to_excel("BAIXA_PROVISORIA.xlsx", index=False)
df_encerrados.to_excel("ENCERRADOS.xlsx", index=False)

print("ETAPA 6 concluída ✔")
print(f"Total de linhas na base: {total_inicial}")
print(f"ATIVOS: {len(df_ativos)}")
print(f"BAIXA PROVISÓRIA: {len(df_baixa)}")
print(f"ENCERRADOS: {len(df_encerrados)}")
print(f"OUTROS STATUS: {len(df_outros)}")

ETAPA 6 concluída ✔
Total de linhas na base: 29452
ATIVOS: 28241
BAIXA PROVISÓRIA: 625
ENCERRADOS: 586
OUTROS STATUS: 0


ETAPA 7 - Gestão das duplicadas

In [10]:
import pandas as pd

# Carregar planilhas
ativos = pd.read_excel("ATIVOS.xlsx")
baixa_prov = pd.read_excel("BAIXA_PROVISORIA.xlsx")
encerrados = pd.read_excel("ENCERRADOS.xlsx")

# Garantir que a data está no formato correto
ativos["Data Cálculo"] = pd.to_datetime(ativos["Data Cálculo"])
baixa_prov["Data Cálculo"] = pd.to_datetime(baixa_prov["Data Cálculo"])
encerrados["Data Cálculo"] = pd.to_datetime(encerrados["Data Cálculo"])

# =========================
# ATIVOS → manter 2 mais recentes com datas diferentes
# =========================
ativos_tratado = (
    ativos.sort_values("Data Cálculo", ascending=False)
          .drop_duplicates(subset=["Pasta", "Data Cálculo"])
          .groupby("Pasta")
          .head(2)
)

# =========================
# BAIXA PROVISÓRIA → manter 1 mais recente
# =========================
baixa_prov_tratado = (
    baixa_prov.sort_values("Data Cálculo", ascending=False)
              .groupby("Pasta")
              .head(1)
)

# =========================
# ENCERRADOS → manter 1 mais recente
# =========================
encerrados_tratado = (
    encerrados.sort_values("Data Cálculo", ascending=False)
              .groupby("Pasta")
              .head(1)
)

# =========================
# SALVAR SOBRESCREVENDO OS ARQUIVOS
# =========================
ativos_tratado.to_excel("ATIVOS.xlsx", index=False)
baixa_prov_tratado.to_excel("BAIXA_PROVISORIA.xlsx", index=False)
encerrados_tratado.to_excel("ENCERRADOS.xlsx", index=False)

print(f"ATIVOS: {len(ativos_tratado)}")
print(f"BAIXA PROVISÓRIA: {len(baixa_prov_tratado)}")
print(f"ENCERRADOS: {len(encerrados_tratado)}")
print("Arquivos atualizados com sucesso ✔")

ATIVOS: 9490
BAIXA PROVISÓRIA: 213
ENCERRADOS: 269
Arquivos atualizados com sucesso ✔


ETAPA 8 - Detecção de Variação nos Ativos

In [11]:
import pandas as pd
import numpy as np

df = pd.read_excel("ATIVOS.xlsx")

df["Data Cálculo"] = pd.to_datetime(df["Data Cálculo"])

# ordenar por processo e data
df = df.sort_values(["Pasta", "Data Cálculo"], ascending=[True, False])

# pegar valor do mês anterior
df["valor_anterior"] = df.groupby("Pasta")["Valor Pedido Objeto Corrigido"].shift(-1)

# manter apenas registros que possuem comparação
comparacao = df[df["valor_anterior"].notna()].copy()

# calcular variação percentual
comparacao["variacao"] = (
    (comparacao["Valor Pedido Objeto Corrigido"] - comparacao["valor_anterior"]) /
    comparacao["valor_anterior"]
)

# identificar casos onde ambos valores são menores que 1
ambos_menor_1 = (
    (comparacao["Valor Pedido Objeto Corrigido"] < 1) &
    (comparacao["valor_anterior"] < 1)
)

# separar classificações
sem_variacao = comparacao[
    (comparacao["variacao"] == 0) | ambos_menor_1
]

com_variacao = comparacao[
    (~ambos_menor_1) &
    (comparacao["variacao"].abs() > 0) &
    (comparacao["variacao"].abs() < 0.05)
]

mudanca_prognostico = comparacao[
    (~ambos_menor_1) &
    (comparacao["variacao"].abs() >= 0.05)
]


# garantir ordenação
sem_variacao = (
    sem_variacao.sort_values("Data Cálculo", ascending=False)
                .groupby("Pasta")
                .head(1)
)

com_variacao = (
    com_variacao.sort_values("Data Cálculo", ascending=False)
                .groupby("Pasta")
                .head(1)
)

mudanca_prognostico = (
    mudanca_prognostico.sort_values("Data Cálculo", ascending=False)
                       .groupby("Pasta")
                       .head(1)
)

# salvar planilhas
sem_variacao.to_excel("ATIVOS_SEM_VARIACAO.xlsx", index=False)
com_variacao.to_excel("ATIVOS_COM_VARIACAO.xlsx", index=False)
mudanca_prognostico.to_excel("MUDANCA_DE_PROGNOSTICO.xlsx", index=False)
print(f"ATIVOS_SEM_VARIACAO: {len(sem_variacao)}")
print(f"ATIVOS_COM_VARIACAO: {len(com_variacao)}")
print(f"MUDANCA_DE_PROGNOSTICO: {len(mudanca_prognostico)}")

ATIVOS_SEM_VARIACAO: 676
ATIVOS_COM_VARIACAO: 4025
MUDANCA_DE_PROGNOSTICO: 36


ETAPA 10 - Criação do Settled

In [12]:
# ETAPA 7 — Criação do SETTLED

baixa = pd.read_excel("BAIXA_PROVISORIA.xlsx")
enc = pd.read_excel("ENCERRADOS.xlsx")

# colunas usadas
col_data = "Data"
col_motivo_bp = "Motivo da Baixa Provisória"
col_data_enc = "Data de Encerramento"
col_motivo_enc = "Motivo Encerramento"

def conv_data(col):
    # 1️⃣ tenta converter normalmente
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")

    # 2️⃣ trata Excel (dias desde 1899)
    mask_excel = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 0) &
        (col_numeric < 60000)
    )

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    # 3️⃣ trata timestamp em nanossegundos
    mask_ns = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 1e12)
    )

    d.loc[mask_ns] = pd.to_datetime(
        col_numeric[mask_ns],
        unit="ns"
    )

    return d

baixa[col_data] = conv_data(baixa[col_data])
enc[col_data] = conv_data(enc[col_data])

print("BAIXA - datas válidas:", baixa[col_data].notna().sum())
print("BAIXA - mínima:", baixa[col_data].min())
print("BAIXA - máxima:", baixa[col_data].max())

print("ENCERRADOS - datas válidas:", enc[col_data].notna().sum())
print("ENCERRADOS - mínima:", enc[col_data].min())
print("ENCERRADOS - máxima:", enc[col_data].max())

# >>> AQUI VOCÊ ALTERA <<<
data_inicio_manual = "26/01/2026"
data_fim_manual = "25/02/2026"

ini = pd.to_datetime(data_inicio_manual, dayfirst=True)
fim = pd.to_datetime(data_fim_manual, dayfirst=True)

baixa_filtrado = baixa[(baixa[col_data] >= ini) & (baixa[col_data] <= fim)].copy()
enc_filtrado = enc[(enc[col_data] >= ini) & (enc[col_data] <= fim)].copy()

# transformação SOMENTE BAIXA PROVISÓRIA
baixa_filtrado[col_data_enc] = baixa_filtrado[col_data]
baixa_filtrado[col_motivo_enc] = baixa_filtrado[col_motivo_bp]

settled = pd.concat([baixa_filtrado, enc_filtrado], ignore_index=True)
settled.to_excel("SETTLED_MENSAL.xlsx", index=False)

print("\nETAPA 7 concluída ✔")
print(f"Período aplicado: {ini.date()} até {fim.date()}")
print(f"BAIXA PROVISÓRIA no período: {len(baixa_filtrado)}")
print(f"ENCERRADOS no período: {len(enc_filtrado)}")
print(f"Total no SETTLED: {len(settled)}")

BAIXA - datas válidas: 213
BAIXA - mínima: 2023-06-14 00:00:00
BAIXA - máxima: 2026-03-03 00:00:00
ENCERRADOS - datas válidas: 113
ENCERRADOS - mínima: 2021-01-01 00:00:00
ENCERRADOS - máxima: 2026-02-26 00:00:00

ETAPA 7 concluída ✔
Período aplicado: 2026-01-26 até 2026-02-25
BAIXA PROVISÓRIA no período: 29
ENCERRADOS no período: 4
Total no SETTLED: 33


ETAPA 12 - Atualização do Histórico

In [13]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

col_pasta = "Pasta"
col_status = "Status"
col_data_cadastro = "Data de cadastro"

# =====================================================
# Ler SETTLED_2.xlsx (novo mês)
# =====================================================

if not os.path.exists("SETTLED_MENSAL.xlsx"):
    print("SETTLED_MENSAL.xlsx não existe. ETAPA 11 ignorada.")

else:

    try:
        novo = pd.read_excel("SETTLED_MENSAL.xlsx", dtype=str)
    except EmptyDataError:
        print("SETTLED_MENSAL.xlsx vazio. ETAPA 11 ignorada.")
        novo = pd.DataFrame()

    if novo.empty:
        print("Sem dados em SETTLED_MENSAL.")
    else:

        novo.columns = novo.columns.astype(str).str.strip()

        # =====================================================
        # Ler histórico
        # =====================================================

        hist = pd.DataFrame()

        if os.path.exists("SETTLED.xlsx"):

            try:
                hist = pd.read_excel("SETTLED.xlsx", dtype=str, engine="openpyxl")
            except Exception:
                print("SETTLED.xlsx vazio ou inválido. Criando histórico novo.")
                hist = pd.DataFrame(columns=novo.columns)

        else:
            hist = pd.DataFrame(columns=novo.columns)

        hist.columns = hist.columns.astype(str).str.strip()

        # =====================================================
        # Garantir colunas essenciais
        # =====================================================

        for col in [col_pasta, col_status, col_data_cadastro]:

            if col not in novo.columns:
                raise ValueError(f"Coluna obrigatória '{col}' não encontrada em SETTLED_2.xlsx")

            if col not in hist.columns:
                hist[col] = None

        # =====================================================
        # Padronizar campos
        # =====================================================

        novo[col_pasta] = novo[col_pasta].astype(str).str.strip()
        hist[col_pasta] = hist[col_pasta].astype(str).str.strip()

        novo[col_data_cadastro] = pd.to_datetime(novo[col_data_cadastro], errors="coerce")
        hist[col_data_cadastro] = pd.to_datetime(hist[col_data_cadastro], errors="coerce")

        # =====================================================
        # Remover duplicados
        # =====================================================

        novo = novo.drop_duplicates(subset=[col_pasta], keep="last")
        hist = hist.drop_duplicates(subset=[col_pasta], keep="last")

        # =====================================================
        # Merge histórico + novo
        # =====================================================

        merged = hist.merge(
            novo,
            on=col_pasta,
            how="outer",
            suffixes=("_hist", "_novo")
        )

        # =====================================================
        # Escolher registro mais recente (corrigir perda de linhas)
        # =====================================================
        novo_existe = (
            merged[f"{col_status}_novo"].notna() |
            merged[f"{col_data_cadastro}_novo"].notna()
        )

        cond_atualizar = novo_existe & (
            (merged[f"{col_status}_hist"] != merged[f"{col_status}_novo"]) |
            (merged[f"{col_data_cadastro}_novo"] > merged[f"{col_data_cadastro}_hist"])
        )

        # linhas que vêm do novo
        atualizar = merged[cond_atualizar]

        # linhas que mantêm histórico
        manter = merged[~cond_atualizar]

        # =====================================================
        # Reconstruir dataframe final (CORRIGIDO: incluir "Pasta")
        # =====================================================

        df_manter = manter[[c for c in merged.columns if c.endswith("_hist")]].rename(
            columns=lambda x: x.replace("_hist", "")
        ).copy()

        df_atualizar = atualizar[[c for c in merged.columns if c.endswith("_novo")]].rename(
            columns=lambda x: x.replace("_novo", "")
        ).copy()

        # Garantir que "Pasta" está presente em ambos os dataframes
        if col_pasta not in df_manter.columns and col_pasta in manter.columns:
            df_manter[col_pasta] = manter[col_pasta]

        if col_pasta not in df_atualizar.columns and col_pasta in atualizar.columns:
            df_atualizar[col_pasta] = atualizar[col_pasta]

        df_atualizado = pd.concat([df_manter, df_atualizar], ignore_index=True)
        df_atualizado = df_atualizado.reset_index(drop=True)

        # =====================================================
        # Salvar de forma segura
        # =====================================================

        temp_file = "SETTLED_temp.xlsx"
        df_atualizado.to_excel(temp_file, index=False)

        if os.path.exists("SETTLED.xlsx"):
            os.remove("SETTLED.xlsx")

        os.rename(temp_file, "SETTLED.xlsx")

        print("ETAPA 11 concluída ✔")
        print("Total histórico:", len(df_atualizado))
        print(f"Coluna 'Pasta' presente: {col_pasta in df_atualizado.columns}")

ETAPA 11 concluída ✔
Total histórico: 34
Coluna 'Pasta' presente: True


ETAPA 13 - Filtro BACENJUD

In [14]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_settled = "SETTLED.xlsx"
arquivo_bacenjud = "BACENJUD.xlsx"

if not os.path.exists(arquivo_settled):
    print("SETTLED.xlsx não existe. Filtro BacenJud ignorado.")

else:
    try:
        settled = pd.read_excel(arquivo_settled)
    except EmptyDataError:
        print("SETTLED.xlsx está vazio. Filtro BacenJud ignorado.")
        settled = pd.DataFrame()

    if settled.empty:
        print("SETTLED sem dados. Filtro BacenJud ignorado.")

    else:

        col_garantia = "Garantia"

        if col_garantia in settled.columns:

            bacenjud = settled[
                settled[col_garantia] == "03 - Penhora Online (BacenJud)"
            ].copy()

            # salvar nova planilha
            bacenjud.to_excel(arquivo_bacenjud, index=False)

            print("Planilha BACENJUD criada.")
            print("Total de casos BacenJud:", len(bacenjud))

        else:
            print("Coluna Garantia não encontrada. Filtro BacenJud ignorado.")

Planilha BACENJUD criada.
Total de casos BacenJud: 1


ETAPA 14 - Pagamentos

In [15]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

def aplicar_pagamentos(nome_arquivo):

    if not os.path.exists(nome_arquivo):
        print(f"{nome_arquivo} não existe. Pagamentos não aplicados.")
        return

    try:
        settled = pd.read_excel(nome_arquivo, dtype=str)
    except EmptyDataError:
        print(f"{nome_arquivo} está vazio. Pagamentos não aplicados.")
        return

    if settled.empty:
        print(f"{nome_arquivo} sem dados. Pagamentos não aplicados.")
        return

    if not os.path.exists("Pagamentos.xlsx"):
        print("Pagamentos.xlsx não encontrado.")
        return

    pag = pd.read_excel("Pagamentos.xlsx", dtype=str)

    if pag.empty:
        print("Arquivo de pagamentos vazio. Nenhuma atualização feita.")
        return

    # =========================
    # LIMPAR COLUNAS
    # =========================

    pag.columns = (
        pag.columns
        .str.replace(r"[\n\r]", " ", regex=True)
        .str.replace('"', '', regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    settled.columns = (
        settled.columns
        .str.replace(r"[\n\r]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    print(f"\n=== PROCESSANDO {nome_arquivo} ===")

    col_pasta_settled = "Pasta"
    col_pasta_pag = "Nº Pasta Projuris"

    col_valor_lancamento = "Valor do Lançamento"
    col_venc_fatal = "VencFatal"
    col_venc_liquid = "VencLíquid"
    col_valor_integral = "Valor integral do Acordo/Condenação"

    if col_pasta_settled not in settled.columns:
        print(f"❌ Coluna '{col_pasta_settled}' não encontrada em {nome_arquivo}")
        return

    if col_pasta_pag not in pag.columns:
        print(f"❌ Coluna '{col_pasta_pag}' não encontrada em Pagamentos.xlsx")
        return

    # =========================
    # SELEÇÃO DE COLUNAS
    # =========================

    colunas_selecionaveis = [col_pasta_pag]

    for col in [
        col_valor_lancamento,
        col_venc_fatal,
        col_venc_liquid,
        col_valor_integral
    ]:
        if col in pag.columns:
            colunas_selecionaveis.append(col)

    pag_reduzido = pag[colunas_selecionaveis].copy()

    # renomear para bater com settled
    pag_reduzido.rename(columns={col_pasta_pag: col_pasta_settled}, inplace=True)

    # =========================
    # MERGE
    # =========================

    settled = settled.merge(
        pag_reduzido,
        on=col_pasta_settled,
        how="left"
    )

    settled.to_excel(nome_arquivo, index=False)

    print(f"✓ Pagamentos aplicados em {nome_arquivo}")
    print(f"Total linhas: {len(settled)}")

# ==========================================================
# RODAR NOS DOIS ARQUIVOS
# ==========================================================

aplicar_pagamentos("SETTLED.xlsx")
aplicar_pagamentos("SETTLED_2.xlsx")


=== PROCESSANDO SETTLED.xlsx ===
✓ Pagamentos aplicados em SETTLED.xlsx
Total linhas: 67

=== PROCESSANDO SETTLED_2.xlsx ===
✓ Pagamentos aplicados em SETTLED_2.xlsx
Total linhas: 65


ETAPA 15 - Macro Assunto

In [16]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

# =========================
# mapa de classificação
# =========================

mapa_macro = {

    "FAR CEF - Vício Construtivo Área Comum": "Construction",
    "Reclamação Proprietário - Vício Construtivo Área Comum": "Construction",
    "Vício Construtivo - Área Comum": "Construction",
    "Atraso de Obra": "Delay",
    "Entrega do Imóvel": "Delay",
    "FAR - Vício Construtivo Unidade Autônoma": "FAR",
    "FAR CEF - Vício Construtivo Unidade Autônoma": "FAR",
    "Acidente de Trabalho": "Labor",
    "Dano Material l Moral": "Cível",
    "Esclarecimento de Informações": "Cível",
    "Horas Extras": "Labor",
    "Reclamação Trabalhista": "Labor",
    "Responsabilidade Solidária/Subsidiária": "Labor",
    "Alienação Fiduciária": "Cível",
    "Ambiental": "Tax",
    "Anulação de Cobrança": "Cível",
    "Arbitragem": "Cível",
    "Cobrança": "Tax",
    "Comissão de Corretagem": "Cível",
    "Corrupção": "Cível",
    "Cota Condominial": "Cível",
    "Desapropriação": "Cível",
    "Descumprimento da Oferta": "Cível",
    "DF Century - Desocupação": "Cível",
    "Embargo de Obra": "Cível",
    "Entrega de Documentos": "Cível",
    "Fornecedor": "Cível",
    "Hipoteca": "Cível",
    "Honorários Advocatícios": "Cível",
    "Incapacidade Financeira": "Cível",
    "Inexistência de Débito": "Cível",
    "Laudêmio": "Tax",
    "Leilão": "Cível",
    "Licenças Públicas": "Cível",
    "Multa Procon": "Tax",
    "Negativação Indevida": "Cível",
    "Obrigação de Fazer/Não Fazer": "Cível",
    "Outorga de Escritura": "Cível",
    "Parceria": "Cível",
    "Percentual Retido Distrato": "Cível",
    "Permutante": "Cível",
    "Posse": "Cível",
    "Protesto - Serviços (água, gás, energia, esgoto)": "Cível",
    "Relação de Consumo": "Cível",
    "Repasse": "Cível",
    "Rescisão Contratual": "Cível",
    "Rescisão Contratual - Locação": "Cível",
    "Restituição de Valores": "Cível",
    "Vizinho": "Cível",
    "Vício Construtivo - Unidade Autônoma": "Cível",
    "IPTU Cliente": "Property Tax",
    "Protesto - IPTU": "Property Tax",
    "CIDE": "Tax",
    "Contribuição Previdenciária": "Tax",
    "CREA": "Tax",
    "CSLL": "Tax",
    "Execução": "Tax",
    "ICMS": "Tax",
    "IE": "Tax",
    "IOF": "Tax",
    "IRPJ": "Tax",
    "ISS": "Tax",
    "ITBI": "Tax",
    "PerDcomp": "Tax",
    "PIS/COFINS": "Tax",
    "Preço Público": "Tax",
    "Protesto": "Tax",
    "Taxa": "Tax",
    "Taxa de Lixo": "Tax",
    "IPTU": "Tax",
    "Castelos D'água": "Cível"
}

# =========================
# arquivos para aplicar
# =========================

arquivos = [
    "SETTLED.xlsx",
    "SETTLED_2.xlsx",   # 👈 ADICIONADO AQUI
    "POS_BP.xlsx",
    "ENTRADAS.xlsx"
]

col_origem = "Assunto Principal"
col_destino = "Macro Assunto"

# =========================
# loop nos arquivos
# =========================

for arquivo in arquivos:

    if not os.path.exists(arquivo):
        print(f"{arquivo} não existe. Tradução ignorada.")
        continue

    try:
        df = pd.read_excel(arquivo)
    except EmptyDataError:
        print(f"{arquivo} está vazio. Tradução ignorada.")
        continue

    if df.empty:
        print(f"{arquivo} sem dados. Tradução ignorada.")
        continue

    df.columns = df.columns.str.strip()

    if col_origem not in df.columns:
        print(f"Coluna '{col_origem}' não encontrada em {arquivo}. Tradução ignorada.")
        continue

    # limpar assunto principal
    df[col_origem] = df[col_origem].astype(str).str.strip()

    # criar coluna Macro Assunto se não existir
    if col_destino not in df.columns:
        df[col_destino] = ""

    # identificar assuntos não classificados
    nao_classificados = df[df[col_origem].map(mapa_macro).isna()][col_origem].dropna().unique()

    if len(nao_classificados) > 0:
        print(f"\nAssuntos sem classificação em {arquivo}:")
        print(sorted(nao_classificados))

    # preencher apenas vazios
    df[col_destino] = df[col_destino].fillna("").astype(str)
    mask = df[col_destino].str.strip() == ""

    df.loc[mask, col_destino] = df.loc[mask, col_origem].map(mapa_macro)

    # salvar
    df.to_excel(arquivo, index=False)

    print(f"Macro Assunto atualizado em {arquivo}")


Assuntos sem classificação em SETTLED.xlsx:
['nan']
Macro Assunto atualizado em SETTLED.xlsx
Macro Assunto atualizado em SETTLED_2.xlsx

Assuntos sem classificação em POS_BP.xlsx:
['Descumprimento de NR', 'FAR - Vício Construtivo Área Comum', 'INSS']
Macro Assunto atualizado em POS_BP.xlsx
Macro Assunto atualizado em ENTRADAS.xlsx


ETAPA 16 - Macro Encerramento

In [17]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

def processar_settled(nome_arquivo):

    if not os.path.exists(nome_arquivo):
        print(f"{nome_arquivo} não existe. Macro encerramento ignorado.")
        return

    try:
        df = pd.read_excel(nome_arquivo)
    except EmptyDataError:
        print(f"{nome_arquivo} está vazio. Macro encerramento ignorado.")
        return

    if df.empty:
        print(f"{nome_arquivo} sem dados. Macro encerramento ignorado.")
        return

    col_macro = "Macro encerramento"
    col_motivo = "Motivo Encerramento"
    col_pasta = "Pasta"
    col_pagamento = "Valor integral do Acordo/Condenação"

    if col_motivo in df.columns and col_pasta in df.columns:

        # criar coluna se não existir
        if col_macro not in df.columns:
            df[col_macro] = ""

        mask_vazio = df[col_macro].isna() | (df[col_macro].astype(str).str.strip() == "")

        df_vazio = df[mask_vazio]
        df_vazio.to_excel(f"Relatorio_Macroencerramento_{nome_arquivo}", index=False)

        valores_settled = [
            "Acordo",
            "Acordo pago - Aguardando baixa",
            "Acordo parcelado- Aguardando baixa",
            "REFIS/PPI - Aguardando baixa"
        ]

        mask_settled = (
            mask_vazio &
            df[col_motivo].isin(valores_settled) &
            ~df[col_motivo].str.contains("terceiro", case=False, na=False)
        )

        df.loc[mask_settled, col_macro] = "Settled"

        valores_lost = [
            "Condenação paga - Aguardando baixa",
            "Procedência"
        ]

        mask_lost = (
            mask_vazio &
            df[col_motivo].isin(valores_lost) &
            ~df[col_motivo].str.contains("terceiro", case=False, na=False)
        )

        df.loc[mask_lost, col_macro] = "Lost"

        if col_pagamento in df.columns:

            mask_restante = (
                df[col_macro].isna() |
                (df[col_macro].astype(str).str.strip() == "")
            )

            mask_won = (
                mask_restante &
                (df[col_pagamento].isna() |
                 (df[col_pagamento].astype(str).str.strip() == ""))
            )

            df.loc[mask_won, col_macro] = "Won"

        sobras = df[
            df[col_macro].isna() |
            (df[col_macro].astype(str).str.strip() == "")
        ]

        if len(sobras) > 0:
            print(f"Arquivos incorretos em {nome_arquivo}:")
            print(sobras[col_pasta].tolist())

        df.to_excel(nome_arquivo, index=False)

        print(f"{nome_arquivo} atualizado com Macro encerramento.")

    else:
        print(f"Colunas necessárias não encontradas em {nome_arquivo}.")


# ==========================================================
# RODAR PARA OS DOIS ARQUIVOS
# ==========================================================

processar_settled("SETTLED.xlsx")
processar_settled("SETTLED_2.xlsx")

Arquivos incorretos em SETTLED.xlsx:
[36806.0, 36806.0, 36806.0, 36806.0, 36806.0, 36806.0, 36806.0, nan, nan]
SETTLED.xlsx atualizado com Macro encerramento.
Arquivos incorretos em SETTLED_2.xlsx:
[36806, 36806, 36806, 36806, 36806, 36806, 36806]
SETTLED_2.xlsx atualizado com Macro encerramento.


ETAPA 17 - Base Histórica

In [18]:
import os
import pandas as pd

def consolidar_base(arquivo_origem, arquivo_destino):

    if not os.path.exists(arquivo_origem):
        print(f"{arquivo_origem} não encontrado.")
        return

    # Ler origem
    origem = pd.read_excel(arquivo_origem, engine="openpyxl")

    if origem.empty:
        print(f"{arquivo_origem} sem dados.")
        return

    # Se destino não existe → cria direto
    if not os.path.exists(arquivo_destino):
        origem.to_excel(arquivo_destino, index=False)
        print(f"{arquivo_destino} criado com {len(origem)} linhas.")
        return

    # Ler destino (uma única vez)
    try:
        destino = pd.read_excel(arquivo_destino, engine="openpyxl")
    except Exception:
        print(f"{arquivo_destino} corrompido. Recriando.")
        origem.to_excel(arquivo_destino, index=False)
        return

    # Concatenar direto (mais rápido)
    consolidado = pd.concat([destino, origem], ignore_index=True)

    consolidado.to_excel(arquivo_destino, index=False)

    print(f"{arquivo_destino} atualizado.")
    print(f"+{len(origem)} linhas adicionadas | Total: {len(consolidado)}")


# Executar
consolidar_base("ENTRADAS.xlsx", "Base_Entradas.xlsx")
consolidar_base("SETTLED.xlsx", "Base_Settled.xlsx")

print("Consolidação concluída ✔")

Base_Entradas.xlsx atualizado.
+18 linhas adicionadas | Total: 1317
Base_Settled.xlsx atualizado.
+67 linhas adicionadas | Total: 201
Consolidação concluída ✔


ETAPA 18 - Assumptions

In [ ]:
import pandas as pd
import os

print("===== GERANDO ASSUMPTIONS_26 =====")

# ==============================
# VALORES FIXOS
# ==============================

# ALTERE AQUI caso os valores mudem no futuro
EXPECTED_LOSS_BASE = 597000
EXPECTED_LOSS_FIXO = 72600

# ALTERE AQUI caso o valor fixo do subtotal mude
SUBTOTAL_FIXO = 72600


# ==============================
# PROCESSOS SEM ATUALIZAÇÃO
# ==============================

sem_atualizacao = 0

if os.path.exists("relatorio_tratado.xlsx"):

    df = pd.read_excel("ATIVOS_SEM_VARIACAO.xlsx")

    AY = df["Valor Pedido Objeto Corrigido"]

    sem_atualizacao = AY.sum()

else:
    print("relatorio_tratado.xlsx não encontrado")


# ==============================
# PROCESSOS COM ATUALIZAÇÃO
# ==============================

com_atualizacao = 0

if os.path.exists("ATIVOS_COM_VARIACAO.xlsx"):

    ativos_var = pd.read_excel("ATIVOS_COM_VARIACAO.xlsx")

    AY = ativos_var["Valor Pedido Objeto Corrigido"]

    com_atualizacao = AY.sum()

else:
    print("ATIVOS_COM_VARIACAO.xlsx não encontrado")


# ==============================
# SETTLED
# ==============================

finally_resolved = 0
savings = 0

if os.path.exists("SETTLED.xlsx"):

    settled = pd.read_excel("SETTLED.xlsx")

    AY = pd.to_numeric(settled["Valor Pedido Objeto Corrigido"], errors="coerce")
    acordo = pd.to_numeric(settled["Valor integral do Acordo/Condenação"], errors="coerce")

    finally_resolved = acordo.sum(skipna=True)

    savings = AY.sum(skipna=True) - finally_resolved

else:
    print("SETTLED.xlsx não encontrado")


# ==============================
# MUDANÇA DE PROGNÓSTICO
# ==============================

mudanca = 0

if os.path.exists("MUDANCA_DE_PROGNOSTICO.xlsx"):

    mud = pd.read_excel("MUDANCA_DE_PROGNOSTICO.xlsx")

    AY = mud["Valor Pedido Objeto Corrigido"]

    mudanca = AY.sum()

else:
    print("MUDANCA_DE_PROGNOSTICO.xlsx não encontrado")


# ==============================
# NEW CLAIMS
# ==============================

new_claims = 0

if os.path.exists("POS_BP.xlsx"):

    entradas = pd.read_excel("POS_BP.xlsx")

    AZ = entradas["Valor Pedido"]

    ativos = entradas["Status"].astype(str).str.upper() == "ATIVOS"

    new_claims = AZ[ativos].sum()

else:
    print("POS_BP.xlsx não encontrado")


# ==============================
# EXPECTED LOSSES
# ==============================

expected_losses_col1 = EXPECTED_LOSS_BASE
expected_losses_col2 = EXPECTED_LOSS_FIXO
expected_losses_col3 = expected_losses_col1 + expected_losses_col2


# ==============================
# SUBTOTAL
# ==============================00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


subtotal_col1 = (
    expected_losses_col1
    + mudanca
    - finally_resolved
    - savings
)

subtotal_col2 = SUBTOTAL_FIXO
subtotal_col3 = subtotal_col1 + subtotal_col2


# ==============================
# EXPECTED LOSSES MENSAL
# ==============================

expected_losses_mensal = subtotal_col1 + new_claims


# ==============================
# DATAFRAME FINAL
# ==============================

assumptions = pd.DataFrame({

    "Descrição": [
        "Processos sem atualização financeira",
        "Processos com atualização financeira ativos",
        "Expected Losses",
        "Finally resolved claims",
        "Savings from finally resolved claims",
        "Changes in expected losses",
        "Subtotal",
        "New Claims",
        "Expected Losses (Mensal)"
    ],

    "Calculo": [
        sem_atualizacao,
        com_atualizacao,
        expected_losses_col1,
        finally_resolved,
        savings,
        mudanca,
        subtotal_col1,
        new_claims,
        expected_losses_mensal
    ],

    "Fixo": [
        "",
        "",
        expected_losses_col2,
        "",
        "",
        "",
        subtotal_col2,
        "",
        ""
    ],

    "Soma": [
        "",
        "",
        expected_losses_col3,
        "",
        "",
        "",
        subtotal_col3,
        "",
        ""
    ]
})


# ==============================
# AJUSTE DE ESCALA (DIVIDIR POR 1000)
# ==============================

linhas_dividir = [
    "Processos sem atualização financeira",
    "Processos com atualização financeira ativos",
    "Finally resolved claims",
    "Savings from finally resolved claims",
    "Changes in expected losses",
    "New Claims",
    "Expected Losses (Mensal)"
]

mask = assumptions["Descrição"].isin(linhas_dividir)




# ==============================
# SALVAR EXCEL
# ==============================

assumptions.to_excel("assumptions_26.xlsx", index=False)

print("assumptions_26.xlsx criado com sucesso")

===== GERANDO ASSUMPTIONS_26 =====
assumptions_26.xlsx criado com sucesso


ETAPA 19 - Macroassuntos nos Ativos

In [22]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

# =========================
# mapa de classificação
# =========================

mapa_macro = {

    "FAR CEF - Vício Construtivo Área Comum": "Construction",
    "Reclamação Proprietário - Vício Construtivo Área Comum": "Construction",
    "Vício Construtivo - Área Comum": "Construction",
    "Atraso de Obra": "Delay",
    "Entrega do Imóvel": "Delay",
    "FAR - Vício Construtivo Unidade Autônoma": "FAR",
    "FAR CEF - Vício Construtivo Unidade Autônoma": "FAR",
    "Acidente de Trabalho": "Labor",
    "Dano Material l Moral": "Cível",
    "Esclarecimento de Informações": "Cível",
    "Horas Extras": "Labor",
    "Reclamação Trabalhista": "Labor",
    "Responsabilidade Solidária/Subsidiária": "Labor",
    "Alienação Fiduciária": "Cível",
    "Ambiental": "Tax",
    "Anulação de Cobrança": "Cível",
    "Arbitragem": "Cível",
    "Cobrança": "Tax",
    "Comissão de Corretagem": "Cível",
    "Corrupção": "Cível",
    "Cota Condominial": "Cível",
    "Desapropriação": "Cível",
    "Descumprimento da Oferta": "Cível",
    "DF Century - Desocupação": "Cível",
    "Embargo de Obra": "Cível",
    "Entrega de Documentos": "Cível",
    "Fornecedor": "Cível",
    "Hipoteca": "Cível",
    "Honorários Advocatícios": "Cível",
    "Incapacidade Financeira": "Cível",
    "Inexistência de Débito": "Cível",
    "Laudêmio": "Tax",
    "Leilão": "Cível",
    "Licenças Públicas": "Cível",
    "Multa Procon": "Tax",
    "Negativação Indevida": "Cível",
    "Obrigação de Fazer/Não Fazer": "Cível",
    "Outorga de Escritura": "Cível",
    "Parceria": "Cível",
    "Percentual Retido Distrato": "Cível",
    "Permutante": "Cível",
    "Posse": "Cível",
    "Protesto - Serviços (água, gás, energia, esgoto)": "Cível",
    "Relação de Consumo": "Cível",
    "Repasse": "Cível",
    "Rescisão Contratual": "Cível",
    "Rescisão Contratual - Locação": "Cível",
    "Restituição de Valores": "Cível",
    "Vizinho": "Cível",
    "Vício Construtivo - Unidade Autônoma": "Cível",
    "IPTU Cliente": "Property Tax",
    "Protesto - IPTU": "Property Tax",
    "CIDE": "Tax",
    "Contribuição Previdenciária": "Tax",
    "CREA": "Tax",
    "CSLL": "Tax",
    "Execução": "Tax",
    "ICMS": "Tax",
    "IE": "Tax",
    "IOF": "Tax",
    "IRPJ": "Tax",
    "ISS": "Tax",
    "ITBI": "Tax",
    "PerDcomp": "Tax",
    "PIS/COFINS": "Tax",
    "Preço Público": "Tax",
    "Protesto": "Tax",
    "Taxa": "Tax",
    "Taxa de Lixo": "Tax",
    "IPTU": "Tax",
    "Castelos D'água": "Cível"
}

# =========================
# arquivos para aplicar
# =========================

arquivos = [
"ATIVOS.xlsx"
]

col_origem = "Assunto Principal"
col_destino = "Macro Assunto"

# =========================
# loop nos arquivos
# =========================

for arquivo in arquivos:

    if not os.path.exists(arquivo):
        print(f"{arquivo} não existe. Tradução ignorada.")
        continue

    try:
        df = pd.read_excel(arquivo)
    except EmptyDataError:
        print(f"{arquivo} está vazio. Tradução ignorada.")
        continue

    if df.empty:
        print(f"{arquivo} sem dados. Tradução ignorada.")
        continue

    df.columns = df.columns.str.strip()

    if col_origem not in df.columns:
        print(f"Coluna '{col_origem}' não encontrada em {arquivo}. Tradução ignorada.")
        continue

    # limpar assunto principal
    df[col_origem] = df[col_origem].astype(str).str.strip()

    # criar coluna Macro Assunto se não existir
    if col_destino not in df.columns:
        df[col_destino] = ""

    # identificar assuntos não classificados
    nao_classificados = df[df[col_origem].map(mapa_macro).isna()][col_origem].dropna().unique()

    if len(nao_classificados) > 0:
        print(f"\nAssuntos sem classificação em {arquivo}:")
        print(sorted(nao_classificados))

    # preencher apenas vazios
    df[col_destino] = df[col_destino].fillna("").astype(str)
    mask = df[col_destino].str.strip() == ""

    df.loc[mask, col_destino] = df.loc[mask, col_origem].map(mapa_macro)

    # salvar
    df.to_excel(arquivo, index=False)

    print(f"Macro Assunto atualizado em {arquivo}")


Assuntos sem classificação em ATIVOS.xlsx:
['Cláusulas Promessa Compra e Venda', 'Cobrança de Tributo Federal', 'Descumprimento de NR', 'FAR - Vício Construtivo Área Comum', 'INSS', 'Procedimento Administrativo', 'Vínculo Empregatício']
Macro Assunto atualizado em ATIVOS.xlsx


ETAPA 20 - Juntar Ativos (com Macro Assunto) e Entradas do Mês

In [23]:
import pandas as pd

# carregar os arquivos
df1 = pd.read_excel("ATIVOS.xlsx")
df2 = pd.read_excel("ENTRADAS.xlsx")


# juntar as bases
df_final = pd.concat([df1, df2], ignore_index=True)

# salvar
df_final.to_excel("BASE_UNIFICADA.xlsx", index=False)

C:\Users\camil\AppData\Local\Temp\ipykernel_2360\2240049591.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df1, df2], ignore_index=True)
